# 🛡️ GRACE Stage 2: Label-Flip Demonstration Sensitivity & Ablation Test

**Môi Trường Thực Thi**: Kaggle GPU / TPU / CPU với Google Cloud Vertex AI MaaS  
**Hợp Đồng Đánh Giá**: `evaluation_contract.yaml`  

### Mục Tiêu Khoa Học:
Đo lường định lượng xem LLM (`publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas`) có thực sự hiểu ngữ nghĩa an ninh trong ví dụ mẫu (**In-Context Demonstration**) hay chỉ đơn thuần sao chép nhãn bề mặt (**Label bias / superficial pattern matching**).

### Thiết Kế Đối Chứng 3 Nhánh (Condition A / B / C):
1. **Condition A (Original Label)**: Target Code + CPG Graph + Top-1 Demo Code + Nhãn Chuẩn ($y_{demo}$)
2. **Condition B (Flipped Label)**: Target Code + CPG Graph + Top-1 Demo Code + Nhãn Đảo ($1 - y_{demo}$)
3. **Condition C (Repeated Original)**: Cùng Prompt với Condition A, chạy lần 2 để đo **Natural Model Variance**

### Các Chỉ Số Đo Lường Chính:
- $\text{Flip Rate} = N(\text{Pred}_A \neq \text{Pred}_B) / N_{valid}$
- $\text{Natural Variance Rate} = N(\text{Pred}_A \neq \text{Pred}_C) / N_{valid}$
- $\text{Net Label Sensitivity} = \text{Flip Rate} - \text{Natural Variance Rate}$
- Phân rã hành vi: Correct $\to$ Wrong (bị nhãn giả lừa) vs Wrong $\to$ Correct (may mắn đoán đúng)

## Bước 1: Thiết Lập Thư Mục Làm Việc (Workspace Setup)
Tìm kiếm mã nguồn dự án trong `/kaggle/input` hoặc sao chép sang `/kaggle/working/GRACE` để có toàn quyền ghi file báo cáo và checkpoints.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

dest_dir = '/kaggle/working/GRACE'
found_src = None

# 1. Đảm bảo luôn lấy mã nguồn mới nhất từ GitHub
if not os.path.exists(dest_dir):
    print('[*] Đang clone mã nguồn GRACE mới nhất từ GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/HuuHieu03/GRACE.git', dest_dir], check=True)
else:
    print('[*] Đang cập nhật mã nguồn mới nhất từ GitHub...')
    subprocess.run(['git', '-C', dest_dir, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', dest_dir, 'reset', '--hard', 'origin/main'], check=True)

subprocess.run(['git', '-C', dest_dir, 'log', '-1', '--oneline'], check=True)

# 2. Chuyển thư mục làm việc và bổ sung sys.path
%cd /kaggle/working/GRACE
for p in ['/kaggle/working/GRACE', '/kaggle/working/GRACE/src']:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'✓ Thư mục làm việc hiện tại: {os.getcwd()}')
!ls -la


## Bước 2: Cài Đặt Các Gói Phụ Thuộc (Dependencies Installation)
Cài đặt các gói cần thiết: `google-auth`, `openai>=1.0.0`, `transformers`, `torch`, `scikit-learn`, `datasets`, `pyyaml`.

In [ ]:
!pip install -q google-auth google-auth-oauthlib openai>=1.0.0 transformers torch scikit-learn datasets pandas tqdm pyyaml
print('✓ [Setup Complete] Các thư viện phụ thuộc đã sẵn sàng!')

## Bước 3: Xác Thực Google Cloud Vertex AI Qua Kaggle Secrets
Đọc Service Account Key từ Kaggle Secret `GCP_SA_KEY`, ghi ra file tạm `/tmp/gcp_sa_key.json`,
thiết lập `GOOGLE_APPLICATION_CREDENTIALS` và kiểm tra sinh Access Token hợp lệ qua `google.auth`.

In [ ]:
import os
import json
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# 1. Đọc Kaggle Secret: GCP_SA_KEY
print('[*] Đang nạp Service Account Key từ Kaggle Secrets...')
try:
    user_secrets = UserSecretsClient()
    sa_json_str = user_secrets.get_secret('GCP_SA_KEY')
    sa_path = Path('/tmp/gcp_sa_key.json')
    with open(sa_path, 'w', encoding='utf-8') as f:
        f.write(sa_json_str.strip())
    os.chmod(sa_path, 0o600)
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(sa_path)
    print(f'✓ [OK] Đã lưu Service Account key tại: {sa_path}')
except Exception as e:
    print(f'[!] Cảnh báo nạp Kaggle Secret: {e}. Vui lòng kiểm tra Add-ons -> Secrets trên Kaggle!')

# 2. Khởi tạo cấu hình và xác thực Vertex AI
from config import config, setup_vertex_credentials
cred_file = setup_vertex_credentials()
print(f'[*] File Credentials hoạt động: {cred_file}')
print(f'[*] Vertex Project ID: {config.vertex_project_id}')
print(f'[*] Vertex Location:   {config.vertex_location}')
print(f'[*] Vertex Model:      {config.vertex_model}')

# 3. Xác thực sinh OAuth2 Token qua google.auth
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
    creds.refresh(Request())
    print(f'✓ [OK] Vertex AI Authentication Verified! Token prefix: {creds.token[:12]}... (valid={creds.valid})')
except Exception as e:
    print(f'[!] Lỗi xác thực Vertex AI: {e}')

## Bước 4: Kiểm Tra Tập Dữ Liệu Devign Clean 100 Samples & Rào Chắn Đồ Thị CPG
Xác nhận tập split `data/splits/devign_dev_clean_100_ids.json` tuân thủ 100% hợp đồng đánh giá:
- 100 mẫu phân tầng (50 Vulnerable, 50 Safe, Seed 42).
- Khẳng định cứng (Hard Assertion): 100% mẫu có `len(nodes) > 0` và `len(edges) > 0` từ Joern 0.2.5 CPG.

In [ ]:
from pathlib import Path
import json

split_path = Path('data/splits/devign_dev_clean_100_ids.json')
assert split_path.exists(), f'Không tìm thấy file split: {split_path}'

with open(split_path, 'r', encoding='utf-8') as f:
    meta = json.load(f)

print(f'[*] Dataset: {meta.get("dataset")} | Split: {meta.get("split_name")}')
print(f'[*] Tổng số mẫu: {meta.get("total_samples")} (Vuln: {meta.get("vulnerable_count")}, Safe: {meta.get("safe_count")})')
print(f'[*] Seed: {meta.get("seed")} | Hash SHA-256: {meta.get("split_id_hash_sha256_short")}')

# Rào chắn kiểm thử cứng theo hợp đồng (Contract Hard Assertion)
sample_details = meta.get("sample_details", [])
assert len(sample_details) == 100, f'Số mẫu chi tiết phải là 100, nhận được: {len(sample_details)}'
for idx, s in enumerate(sample_details):
    nodes = s.get("nodes", [])
    edges = s.get("edges", [])
    assert len(nodes) > 0, f'LỖI RÀO CHẮN: Mẫu {s.get("id")} (chỉ số {idx}) thiếu nodes đồ thị CPG!'
    assert len(edges) > 0, f'LỖI RÀO CHẮN: Mẫu {s.get("id")} (chỉ số {idx}) thiếu edges đồ thị CPG!'

print(f'✓ [OK] RÀO CHẮN HỢP ĐỒNG ĐẠT CHUẨN: 100% ({len(sample_details)}/100) mẫu có đầy đủ CPG nodes và edges.')


## Bước 5: Thực Thi Thí Nghiệm Đảo Nhãn (Run Label-Flip Test)
Gọi `scripts/run_label_flip_test.py` với mô hình Vertex AI:
- Backbone: `publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas`
- Tham số: `temperature=0.0`, `seed=42`, `max_tokens=256`
- Context: `graph=True`, `context_limit=8192`, `graph_nodes_max=25`, `graph_edges_max=25`
- Chính sách Retry: `max_attempts=5`, `exponential_backoff=5s..120s`
- Kết quả lưu tại: `output/stage2/label_flip/label_flip_test_report.json`

In [ ]:
!python scripts/run_label_flip_test.py \
    --dev_split data/splits/devign_dev_clean_100_ids.json \
    --dataset devign \
    --model publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas \
    --provider "Vertex AI" \
    --location us-south1 \
    --output output/stage2/label_flip/label_flip_test_report.json

# Sao chép kết quả về thư mục gốc /kaggle/working để đảm bảo Kaggle nén xuất đầy đủ
!mkdir -p /kaggle/working/output/stage2/label_flip
!cp output/stage2/label_flip/label_flip_test_report.json /kaggle/working/output/stage2/label_flip/ 2>/dev/null || true
!cp output/stage2/label_flip/label_flip_test_report.json /kaggle/working/label_flip_test_report.json 2>/dev/null || true


## Bước 6: Phân Tích & Nghiệm Thu Kết Quả Thực Nghiệm
Đọc file báo cáo JSON, tổng hợp định lượng các chỉ số Flip Rate, Natural Variance, Net Label Sensitivity và so sánh hiệu năng giữa 3 điều kiện A/B/C.

In [ ]:
import json
from pathlib import Path

report_file = Path('output/stage2/label_flip/label_flip_test_report.json')
if not report_file.exists():
    report_file = Path('output/label_flip_test_report.json')
if not report_file.exists():
    report_file = Path('/kaggle/working/output/stage2/label_flip/label_flip_test_report.json')
if not report_file.exists():
    report_file = Path('/kaggle/working/label_flip_test_report.json')

if report_file.exists():
    with open(report_file, 'r', encoding='utf-8') as f:
        rep = json.load(f)
    
    qf = rep.get('quantitative_findings', {})
    bb = rep.get('behavioral_breakdown', {})
    cm = rep.get('condition_metrics', {})
    ca = cm.get('condition_a_original', {})
    cb = cm.get('condition_b_flipped', {})
    cc = cm.get('condition_c_repeated', {})
    
    print('=' * 80)
    print('  TỔNG HỢP KẾT QUẢ THÍ NGHIỆM ĐẢO NHÃN TRÊN VERTEX AI (QWEN-3 CODER 480B)')
    print('=' * 80)
    print(f'[*] Quy mô kiểm thử: {rep.get("valid_samples")} / {rep.get("total_samples")} mẫu hợp lệ')
    print(f'[*] Tỷ lệ Đảo Nhãn (Flip Rate):           {qf.get("flip_count")} / {rep.get("valid_samples")} ({qf.get("flip_rate_pct")}%)')
    print(f'[*] Dao động Tự Nhiên (Natural Variance): {qf.get("natural_variance_count")} / {rep.get("valid_samples")} ({qf.get("natural_variance_pct")}%)')
    print(f'[*] ĐỘ NHẠY NHÃN THỰC CHẤT (Net Sens.):   {qf.get("net_label_sensitivity_pct")}%')
    print('-' * 80)
    print(f'[*] Phân rã hành vi:')
    print(f'    - Bị nhãn giả lừa (Correct -> Wrong): {bb.get("correct_to_wrong")} mẫu ({bb.get("correct_to_wrong_pct")}%)')
    print(f'    - May mắn đoán trúng (Wrong -> Correct): {bb.get("wrong_to_correct")} mẫu ({bb.get("wrong_to_correct_pct")}%)')
    print(f'    - Kiên định đúng (Stayed Correct):    {bb.get("stayed_correct")} mẫu ({bb.get("stayed_correct_pct")}%)')
    print(f'    - Kiên định sai (Stayed Wrong):       {bb.get("stayed_wrong")} mẫu ({bb.get("stayed_wrong_pct")}%)')
    print('-' * 80)
    print(f'[*] Hiệu năng phân loại theo điều kiện:')
    print(f'    - Condition A (Original): Accuracy={ca.get("accuracy")} | Precision={ca.get("precision")} | Recall={ca.get("recall")} | F1={ca.get("f1_score")} | MCC={ca.get("mcc")}')
    print(f'    - Condition B (Flipped):  Accuracy={cb.get("accuracy")} | Precision={cb.get("precision")} | Recall={cb.get("recall")} | F1={cb.get("f1_score")} | MCC={cb.get("mcc")}')
    print(f'    - Condition C (Repeated): Accuracy={cc.get("accuracy")} | Precision={cc.get("precision")} | Recall={cc.get("recall")} | F1={cc.get("f1_score")} | MCC={cc.get("mcc")}')
    print('=' * 80)
else:
    print('[!] Không tìm thấy file output/stage2/label_flip/label_flip_test_report.json.')